---
title: "À quelques kilomètres près : lycées et géographie des cadres dans trois aires urbaines de province"
output-file: index.html
---

In [ ]:
# Le SSPCloud définit MPLBACKEND=Agg : sans cette ligne, les figures ne s'affichent pas
%matplotlib inline
import warnings

from IPython.display import display

from cadres_lycees import analyse, collecte, config, figures, geo, nettoyage, pipeline

# Sans BUCKET_S3, stockage.py avertit qu'il lit et écrit dans data/ : c'est le cas prévu d'un
# compte neuf (README). Les autres avertissements, dont le repli d'une copie gelée, restent
warnings.filterwarnings("ignore", message="Variables d'environnement manquantes")

## Résumé

## 1. Introduction

### 1.1 Pourquoi les lycées

### 1.2 Question et hypothèses

### 1.3 Ce que le projet peut et ne peut pas montrer

## 2. Construire la base : une ligne par commune, quatre familles de variables

### 2.1 Schéma du pipeline et tableau des sources

In [ ]:
# Le rapport reconstruit la base depuis les copies gelées (PLAN §4, étape G) : final/ n'est
# qu'une sortie. Les fichiers lus et leur sha256 sont listés en annexe
base, lycees, paires = pipeline.construire_base(config.AIRES_RETENUES)

### 2.2 Variable à expliquer

### 2.3 Variable d'intérêt : l'accessibilité aux lycées

In [ ]:
# Accès des communes aux lycées et aux autres infrastructures, aire par aire (médianes)
figures.tableau_acces(analyse.resume_acces(base))

### 2.4 Contrôles et mauvais contrôles

### 2.5 Encadré « choix et biais »

In [ ]:
# Fiabilité de l'IVAL (§3.2, A7) : seule la ligne « ensemble » tranche H3 (seuil de 0,5)
fiabilite = analyse.fiabilite_ival(base, lycees, collecte.ival_lycees())
figures.tableau_fiabilite(fiabilite)

## 3. Trois aires, trois profils

### 3.1 Pourquoi ces trois aires

In [ ]:
# La Fig. 1 montre les 8 candidates pour justifier le choix ; la base du rapport (section 2) ne
# porte que sur les 3 aires retenues
referentiel_candidates = geo.referentiel_communes(config.AIRES_CANDIDATES)
lycees_8_aires = pipeline.lycees_avec_aire(referentiel_candidates)
profil = analyse.profil_aires(lycees_8_aires)

In [ ]:
#| column: page
# Titre provisoire : le message de la figure est écrit par le groupe
figures.fig1(profil, lycees_8_aires, "Les 8 aires candidates", config.AIRES_RETENUES)

In [ ]:
#| output: asis
# Le tableau de support du choix des aires (§2.2, §5.2), commité avant le choix (commit 0583839),
# replié : il justifie le choix sans alourdir la lecture
print('::: {.callout-note collapse="true" title="Tableau de support du choix des aires"}')
print(figures.document_markdown(config.CHEMIN_TABLEAU_SUPPORT))
print(":::")

### 3.2 Le gradient centre-périphérie des cadres

In [ ]:
#| output: asis
# Part de cadres de 1999 à 2022, de la série historique du recensement ramenée au COG 2025 ;
# une aire par onglet, comme la Fig. 3, et un curseur pour choisir le recensement
series_csp = {annee: collecte.serie_csp_2554(annee) for annee in config.GEOGRAPHIE_SERIE_CSP}
parts_dates = nettoyage.parts_cadres_dates(series_csp, collecte.passage_communes(), base)
print("::: {.panel-tabset}")
for aire in config.AIRES_RETENUES:
    print(f"\n#### {aire}\n")
    # Titres provisoires : le message de chaque carte est écrit par le groupe
    communes_aire = base[base["aire"] == aire]
    display(figures.carte_cadres_dates(communes_aire, parts_dates, f"Aire de {aire}"))
print(":::")

### 3.3 La géographie de l'accès aux lycées

In [ ]:
#| output: asis
# Une carte par aire retenue, chacune dans un onglet : les lignes « ::: » et « #### » écrites
# ici sont le balisage Markdown des onglets de Quarto
print("::: {.panel-tabset}")
for aire in config.AIRES_RETENUES:
    print(f"\n#### {aire}\n")
    # Titres provisoires et descriptifs : le message de chaque carte est écrit par le groupe
    display(figures.fig3(base[base["aire"] == aire], lycees, f"Aire de {aire}"))
print(":::")

## 4. À position égale, l'accès aux lycées fait-il une différence ?

### 4.1 Comparaison par anneaux

In [ ]:
# H1 : pentes des paires, principale sur l'écart de distance au lycée privé et secondaire sur
# l'écart du nombre de lycées privés à 5 km (§5.3), avec les mêmes tirages du bootstrap ; écarts
# des cellules d'anneaux à 5 et 10 km (§5.2). La Fig. 4 sert aussi H2 (§5.4)
pentes_principales = analyse.pentes_paires_par_aire(paires, "d_dist_lyc_prive_min")
pentes_secondaires = analyse.pentes_paires_par_aire(paires, "d_n_lyc_prive_5km")
anneaux = analyse.ecarts_anneaux(base)
# Titre provisoire : le message de la figure est écrit par le groupe
figures.fig4(pentes_principales, pentes_secondaires, anneaux, "Estimations de H1 par aire")

### 4.2 Comparaison entre communes contiguës

In [ ]:
# Titre provisoire : le message de la figure est écrit par le groupe
figures.fig5(paires, pentes_principales, "Écarts de part de cadres et de distance au lycée privé")

### 4.3 L'Ouest est-il différent ?

### 4.4 Qualité ou présence ?

In [ ]:
# H3 (§5.5, A16) : population des communes ayant un lycée retenu à 5 km, régression sur l'IVAL
# et le nombre de lycées à IVAL, centrés réduits dans chaque aire, bootstrap par grappes
population_h3 = analyse.population_h3(base, lycees)
estimation_h3 = analyse.estimation_h3(population_h3)
# Titre provisoire : le message de la figure est écrit par le groupe
display(figures.fig6(analyse.terciles_h3(population_h3), "Évolution de la part de cadres et IVAL"))
# β en points de part de cadres pour un écart-type ; seule la portée « ensemble » porte le verdict
figures.tableau_h3(estimation_h3)

### 4.5 Robustesse : tableau récapitulatif et indice de Moran

In [ ]:
# Robustesse (§5.6) : pente principale de H1 dans chaque variante, en points par km (par lycée
# pour les pentes secondaires), avec les mêmes estimation et bootstrap que la spécification
# principale
robustesse = analyse.robustesse_h1(
    base, paires, lycees, pipeline.lycees_prives_robustesse(), collecte.chefs_lieux()
)
display(figures.tableau_robustesse(robustesse))
# Robustesse de H3 (§3.2) : période de l'IVAL et indicateur de valeur ajoutée, 3 aires ensemble
display(figures.tableau_h3(analyse.robustesse_h3(base, lycees, collecte.ival_lycees())))
# Indice I de Moran de la part de cadres, puis de l'écart à la moyenne de son anneau
figures.tableau_moran(analyse.moran_part_cadres(base))

## 5. Les lycées face aux autres infrastructures

In [ ]:
# H4 : pentes des paires pour chaque mesure d'accès, centrée réduite dans chaque aire (§5.7)
pentes_h4 = analyse.pentes_h4(base, paires)
# Titre provisoire : le message de la figure est écrit par le groupe
figures.fig7(pentes_h4, "Lycées et autres infrastructures")

## 6. Conclusion

In [ ]:
# Tableau des hypothèses, par les règles fixées avant les résultats : H1 par aire (§5.3, A4,
# A13), H3 toutes aires ensemble (§5.5, A16) ; H2 et H4 descriptives (§5.4, A20)
verdicts_h1 = analyse.verdict_h1(pentes_principales, pentes_secondaires, anneaux)
verdict_h3 = analyse.verdict_h3(estimation_h3, fiabilite.loc["ensemble", "r_spearman_brown"])
figures.tableau_hypotheses(verdicts_h1, verdict_h3)

In [ ]:
#| output: asis
# Limites : part de cadres de 15 à 64 ans par IRIS en 2011 et 2022 (A18), exploration sans test
# ni lycées. Les communes-centres sont à 0 km du centre ; à Lyon, l'arrondissement le plus proche
# du centre a pour commune parente 69123, le code des contours IRIS. Une aire par onglet, comme
# la Fig. 3
parts_iris = nettoyage.part_cadres_iris(
    collecte.activite_iris_2011(), collecte.activite_iris_2022(), collecte.passage_communes()
)
print("::: {.panel-tabset}")
for aire, code_aav in config.AIRES_RETENUES.items():
    print(f"\n#### {aire}\n")
    communes_aire = base[base["aire"] == aire]
    commune_centre = communes_aire.loc[communes_aire["dist_centre_km"].idxmin(), "code_parent"]
    # Titres provisoires : le message de chaque carte est écrit par le groupe
    display(
        figures.carte_iris(
            collecte.contours_iris(code_aav), parts_iris, commune_centre, f"Aire de {aire}"
        )
    )
print(":::")

## 7. Annexes

### Reproduire le projet

In [ ]:
# Fichiers bruts lus par le rapport, avec le sha256 contrôlé à chaque lecture (README)
pipeline.fichiers_bruts()

### Dictionnaire des variables

In [ ]:
#| output: asis
print(figures.document_markdown(config.CHEMIN_DICTIONNAIRE))

### Journal des décisions

In [ ]:
#| output: asis
# Le journal du plan (§13), daté, tel qu'il est dans le dépôt ; replié, car il est plus long que
# le reste du rapport
print('::: {.callout-note collapse="true" title="Journal des décisions"}')
print(figures.document_markdown(config.CHEMIN_PLAN, config.TITRE_JOURNAL_PLAN))
print(":::")